# Stage 2 — Baseline Activation Extraction & Refusal Behavior

For each of the 4 base (undefended) models: extract last-token residual-stream activations at every layer for harmful and harmless prompts in every data split, and confirm baseline refusal behavior on harmful prompts.

Follows `claude.md`: one model on GPU at a time, checkpointed saves to Drive, activations never pushed to GitHub (Drive only).

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets scikit-learn matplotlib

## 1. Mount Drive (claude.md rule 1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/RepEng_Survival/results'
ACTIVATIONS_DIR = '/content/drive/MyDrive/RepEng_Survival/results/activations'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ACTIVATIONS_DIR, exist_ok=True)

## 2. Clone/pull repo (claude.md rule 1)

In [ ]:
REPO_URL = 'https://github.com/Pranamya16/RepEng_Survival.git'

if not os.path.exists('/content/RepEng_Survival'):
    !git clone {REPO_URL} /content/RepEng_Survival
else:
    !cd /content/RepEng_Survival && git pull

%cd /content/RepEng_Survival

import sys
sys.path.append('/content/RepEng_Survival/src')

DATA_DIR = '/content/RepEng_Survival/data'

## 3. Anti-disconnect (claude.md rule 2)

In [ ]:
from IPython.display import Javascript

display(Javascript('''
function KeepAlive(){
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(KeepAlive, 60000)
'''))

## 4. Hugging Face login (Gemma-2 and Llama-3.2 are gated)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN'))

## 5. Checkpoint helpers (claude.md rule 4)

In [ ]:
import json

def save_result(name, data, directory=RESULTS_DIR):
    path = os.path.join(directory, name)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def load_result_if_exists(name, directory=RESULTS_DIR):
    path = os.path.join(directory, name)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

## 6. Load data splits

In [ ]:
SPLIT_NAMES = ['probe_train', 'probe_test', 'steering_construction', 'recovery_evaluation']

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

splits = {name: load_jsonl(os.path.join(DATA_DIR, f'{name}.jsonl')) for name in SPLIT_NAMES}
for name, rows in splits.items():
    print(f'{name}: {len(rows)} rows')

## 7. Extraction + baseline refusal check — one model at a time (claude.md rule 3)
For each model: for each split, extract last-token activations at every layer for all prompts (saved to Drive as one `.pt` per model/split), then generate short responses to the harmful prompts only and classify refusal (saved as a JSON summary). Resumable per model/split.

In [ ]:
import torch
import gc
import traceback
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from extraction import extract_activations, save_activations
from metrics import refusal_classifier

MODELS = {
    'gemma-2-2b-it': 'google/gemma-2-2b-it',
    'llama-3.2-3b-instruct': 'meta-llama/Llama-3.2-3B-Instruct',
    'qwen2.5-3b-instruct': 'Qwen/Qwen2.5-3B-Instruct',
    'phi-4-mini-instruct': 'microsoft/Phi-4-mini-instruct',
}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

MAX_NEW_TOKENS = 40

for short_name, hf_id in MODELS.items():
    print(f'=== {short_name} ===')
    model = None
    tokenizer = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            hf_id, quantization_config=bnb_config, device_map='auto'
        )

        for split_name, rows in splits.items():
            act_path = os.path.join(ACTIVATIONS_DIR, f'{short_name}_{split_name}.pt')
            refusal_result_name = f'stage2_refusal_{short_name}_{split_name}.json'

            existing_refusal = load_result_if_exists(refusal_result_name)
            if os.path.exists(act_path) and existing_refusal:
                print(f'  Skipping {split_name}, already done.')
                continue

            print(f'  Extracting activations: {split_name} ({len(rows)} prompts)')
            formatted = [
                tokenizer.apply_chat_template(
                    [{'role': 'user', 'content': row['prompt']}],
                    tokenize=False, add_generation_prompt=True,
                )
                for row in rows
            ]
            acts = extract_activations(model, tokenizer, formatted)
            payload = {
                'activations': acts,
                'labels': [row['label'] for row in rows],
                'sources': [row['source'] for row in rows],
                'prompts': [row['prompt'] for row in rows],
            }
            save_activations(payload, act_path)

            print(f'  Checking baseline refusal on harmful prompts in {split_name}')
            harmful_rows = [row for row in rows if row['label'] == 'harmful']
            refused = 0
            samples = []
            for row in harmful_rows:
                inputs = tokenizer.apply_chat_template(
                    [{'role': 'user', 'content': row['prompt']}],
                    add_generation_prompt=True, return_tensors='pt', return_dict=True,
                ).to(model.device)
                output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
                generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                is_refusal = refusal_classifier(generated)
                refused += int(is_refusal)
                if len(samples) < 5:
                    samples.append({'prompt': row['prompt'], 'generated': generated, 'refused': is_refusal})

            refusal_result = {
                'model': short_name,
                'split': split_name,
                'n_harmful': len(harmful_rows),
                'n_refused': refused,
                'refusal_rate': refused / len(harmful_rows) if harmful_rows else None,
                'samples': samples,
            }
            save_result(refusal_result_name, refusal_result)
            print(f"    refusal_rate = {refusal_result['refusal_rate']}")

    except Exception as e:
        tb = traceback.format_exc()
        print(f'FAILED on {short_name}: {e!r}')
        print(tb)
        save_result(f'stage2_error_{short_name}.json', {'model': short_name, 'error': str(e) or repr(e), 'traceback': tb})

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

## 8. Baseline refusal rate table

In [ ]:
import pandas as pd

rows = []
for short_name in MODELS:
    for split_name in SPLIT_NAMES:
        r = load_result_if_exists(f'stage2_refusal_{short_name}_{split_name}.json')
        if r:
            rows.append({'model': short_name, 'split': split_name, 'n_harmful': r['n_harmful'], 'refusal_rate': r['refusal_rate']})

table = pd.DataFrame(rows)
table

## 9. Results stay on Drive (claude.md rule 5)
Activation tensors and refusal-rate JSONs live in `RESULTS_DIR` / `ACTIVATIONS_DIR` on Drive only — nothing here gets pushed to GitHub.

## Deliverable check
- [ ] Activation tensors saved to Drive for all 4 models × 4 splits
- [ ] Baseline refusal rate table computed for all 4 models
- [ ] Refusal rates are high (models are refusing before any defense is applied) — if not, flag before moving to Stage 3